# hai07 - ukur kecepatan latih di Kaggle GPU T4 x2 (DDP dua GPU)
Mengukur **satu GPU** dan **dua GPU** (`torchrun`) pada konfigurasi yang sama, sehingga efisiensi penskalaan terlihat dari angka, bukan perkiraan.

**Persiapan**
1. Add Input: dataset **hai07-bench** (unggah ulang `kaggle/hai07-bench.zip` terbaru; dibuat oleh `python tools/package_bench_kit.py`).
2. Settings: Accelerator = **GPU T4 x2**, Internet tidak perlu.
3. Jalankan sel 1 lalu sel 2 (uji asap, berhenti bila gagal), baru sel 3 dan 4.

Setiap konfigurasi berjalan sebagai proses terpisah (kehabisan memori tidak menghentikan yang lain). Perkiraan total sel 3: ±10-20 menit.
Kode DDP ini baru diuji dengan dua proses CPU (gloo); **belum pernah di dua GPU sungguhan**. Bila ada galat, kirim 15 baris terakhir keluarannya.

In [ ]:
import glob, os, subprocess, sys, zipfile, json
WORK = "/kaggle/working/hai07-bench"; os.makedirs(WORK, exist_ok=True)
src_dir = next((os.path.dirname(os.path.dirname(p)) for p in glob.glob("/kaggle/input/**/tools/bench_train.py", recursive=True)), None)
if src_dir: subprocess.run(["cp", "-r", src_dir + "/.", WORK], check=True)
else:
    z = glob.glob("/kaggle/input/**/hai07-bench.zip", recursive=True); assert z, "tambahkan dataset hai07-bench sebagai Input"
    zipfile.ZipFile(z[0]).extractall(WORK)
os.chdir(WORK)
import torch
n = torch.cuda.device_count()
print("torch", torch.__version__, "| jumlah GPU:", n, [torch.cuda.get_device_name(i) for i in range(n)])
assert n >= 2, "pilih Accelerator GPU T4 x2 (hanya %d GPU terlihat)" % n
src = open("tools/bench_train.py", encoding="utf-8").read()
assert "WORLD_SIZE" in src, "zip lama: unggah hai07-bench.zip terbaru (bench_train.py harus mendukung DDP)"


In [ ]:
import os, subprocess, sys, json
WORK = "/kaggle/working/hai07-bench"
assert os.path.exists(WORK + "/tools/bench_train.py"), "jalankan sel 1 dulu (menyiapkan berkas)"
TORCHRUN = [sys.executable, "-u", "-m", "torch.distributed.run", "--standalone", "--nproc_per_node=2"]
def bench(extra, gpus, out=None, timeout=1500):
    cmd = (TORCHRUN if gpus == 2 else [sys.executable, "-u"]) + [WORK + "/tools/bench_train.py", "--device", "cuda", "--steps", "8", "--warmup", "3"] + (["--out", out] if out else []) + extra
    p = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, cwd=WORK)
    return p, [l for l in p.stdout.splitlines() if l.startswith("{")]
# uji asap: 2 GPU, model kecil
p, last = bench(["--preset", "tiny", "--ctx", "64", "--batch", "2", "--amp"], 2, timeout=600)
print(p.stdout[-1500:]); print(p.stderr[-2500:])
assert last and json.loads(last[-1]).get("jumlah_gpu") == 2, "UJI ASAP DDP GAGAL: kirim keluaran di atas; jangan lanjut ke sel 3"
print("UJI ASAP BERHASIL: DDP 2 GPU jalan")


In [ ]:
# (nama, argumen bersama). Tiap konfigurasi dijalankan dengan 1 GPU lalu 2 GPU. --batch = PER GPU.
COMMON = ["--amp", "--fused-adamw"]
RUNS = [
    ("100M ctx256 batch32",            ["--preset", "100m", "--ctx", "256",  "--batch", "32"]),
    ("100M ctx1024 batch8",            ["--preset", "100m", "--ctx", "1024", "--batch", "8"]),
    ("300M ctx1024 ckpt batch4",       ["--preset", "300m", "--ctx", "1024", "--batch", "4", "--grad-checkpoint"]),
    ("540M ctx1024 ckpt batch2",       ["--preset", "540m", "--ctx", "1024", "--batch", "2", "--grad-checkpoint"]),
    ("540M ctx1024 ckpt batch4",       ["--preset", "540m", "--ctx", "1024", "--batch", "4", "--grad-checkpoint"]),
]
OUT = "/kaggle/working/bench_t4x2.jsonl"
if os.path.exists(OUT): os.remove(OUT)
rows = []
for name, extra in RUNS:
    for gpus in (1, 2):
        label = f"{name} | {gpus} GPU"; print("=" * 90); print(label, flush=True)
        try:
            p, last = bench(COMMON + extra, gpus, out=OUT)
            if last:
                d = json.loads(last[-1]); d["nama"] = label; d["gpus"] = gpus; rows.append(d); print(last[-1])
            else:
                err = (p.stderr.strip().splitlines() or ["(tanpa keluaran)"])[-1][:200]
                rows.append({"nama": label, "gpus": gpus, "galat": err}); print("GALAT:", err)
        except subprocess.TimeoutExpired:
            rows.append({"nama": label, "gpus": gpus, "galat": "melewati batas waktu"}); print("GALAT: batas waktu")


In [ ]:
by = {}
for r in rows:
    by.setdefault(r["nama"].split(" | ")[0], {})[r["gpus"]] = r
for r in rows:   # efisiensi penskalaan 2 GPU terhadap 1 GPU (100% = linear)
    one = by[r["nama"].split(" | ")[0]].get(1, {})
    if r["gpus"] == 2 and "token_per_detik" in r and "token_per_detik" in one:
        r["skala_persen"] = round(100 * r["token_per_detik"] / (2 * one["token_per_detik"]))
cols = ["nama", "parameter_juta", "token_per_detik", "token_per_detik_per_gpu", "skala_persen", "mfu_persen", "memori_puncak_gb", "jam_untuk_10_miliar_token", "galat"]
cols = [c for c in cols if any(c in r for r in rows)]
w = {c: max(len(c), *(len(str(r.get(c, "-"))) for r in rows)) for c in cols}
print("  ".join(c.ljust(w[c]) for c in cols)); print("  ".join("-" * w[c] for c in cols))
for r in rows: print("  ".join(str(r.get(c, "-")).ljust(w[c]) for c in cols))
print(chr(10) + "skala_persen: 100 = dua GPU tepat dua kali satu GPU. Kaggle T4 tanpa NVLink (PCIe), jadi di bawah 100 wajar.")
print("jam_untuk_10_miliar_token x 1,08 = jam latih 540M (10,8 miliar token). Kuota Kaggle GPU per minggu terbatas: cek di profil akun.")
print("Salin tabel ini dan kirim ke pengembang. Berkas lengkap:", OUT)
